# Baseline Prioritization Methods

These methods are intentionally not machine learning techniques. They provide baseline comparisons for CVSS-only, EPSS-only, KEV-first, and a simple weighted score.

In [ ]:
# Setup cell for Colab compatibility.
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()

if "google.colab" in sys.modules:
    repo_url = os.getenv("GITHUB_REPO_URL")
    if repo_url:
        !git clone {repo_url} /content/cyberguard-ai
        %cd /content/cyberguard-ai
        REPO_ROOT = Path.cwd()

for candidate in [REPO_ROOT, Path("/content/cyberguard-ai"), Path("/workspace"), Path("/content")]:
    if (candidate / "data" / "processed" / "risk_features.csv").exists():
        REPO_ROOT = candidate
        break

os.chdir(REPO_ROOT)
print(f"Project root: {REPO_ROOT}")

!python -m pip install -q pandas numpy scikit-learn matplotlib plotly

from src.baselines.cvss_baseline import cvss_only_baseline
from src.baselines.epss_baseline import epss_only_baseline
from src.baselines.kev_baseline import kev_first_baseline
from src.baselines.weighted_baseline import weighted_baseline

import pandas as pd

risk_df = pd.read_csv(REPO_ROOT / "data" / "processed" / "risk_features.csv")
print(risk_df.shape)

In [ ]:
# Generate baseline outputs.
cvss_df = cvss_only_baseline(risk_df)
epss_df = epss_only_baseline(risk_df)
kev_df = kev_first_baseline(risk_df)
weighted_df = weighted_baseline(risk_df)

print(cvss_df.head())
print(epss_df.head())
print(kev_df.head())
print(weighted_df.head())

In [ ]:
# Combine outputs into a consistent format.
all_baselines = pd.concat([cvss_df, epss_df, kev_df, weighted_df], ignore_index=True)
print(all_baselines.head())

output_path = REPO_ROOT / "outputs" / "baseline_results.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
all_baselines.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

## Methodology summary

1. CVSS-only baseline: ranks by CVSS score.
2. EPSS-only baseline: ranks by EPSS score.
3. KEV-first baseline: KEV vulnerabilities are ranked highest, then CVSS and EPSS are used as tie-breakers.
4. Weighted baseline: simple weighted score using CVSS, EPSS, and KEV evidence.

These are baseline heuristics, not ML models, and they are used only as a comparison layer.